# 📦 Olist E-Commerce Dataset — Data Loading & Initial Exploration

This notebook focuses on loading and understanding the structure of the Brazilian E-Commerce dataset provided by Olist.

## 🎯 Objectives
- Load all raw datasets
- Understand schema and relationships
- Perform initial data inspection
- Identify data quality issues
- Generate early business insights

This step is important before moving into data cleaning, feature engineering, and building an analytics agent.

In [7]:
import numpy as np
import pandas as pd
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

In [8]:
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")

In [ ]:
files = os.listdir(DATA_DIR)

dataframes = {}

for file in files:
    if file.endswith(".csv"):
        file_path = os.path.join(DATA_DIR, file)
        df_name = file.replace(".csv", "")
        
        dataframes[df_name] = pd.read_csv(file_path)
        
        print(f"Loaded {df_name} with shape {dataframes[df_name].shape}")

Loaded olist_sellers_dataset with shape (3095, 4)
Loaded product_category_name_translation with shape (71, 2)
Loaded olist_orders_dataset with shape (99441, 8)
Loaded olist_order_items_dataset with shape (112650, 7)
Loaded olist_customers_dataset with shape (99441, 5)
Loaded olist_geolocation_dataset with shape (1000163, 5)
Loaded olist_order_payments_dataset with shape (103886, 5)
Loaded olist_order_reviews_dataset with shape (99224, 7)
Loaded olist_products_dataset with shape (32951, 9)


## 📂 Dataset Overview

All CSV files have been successfully loaded into pandas DataFrames.

### Key Tables:
- **orders** → central transaction table
- **order_items** → item-level details for each order
- **products** → product metadata
- **customers** → customer information
- **payments** → payment transactions
- **reviews** → customer feedback
- **sellers** → seller information
- **geolocation** → location data
- **category_translation** → category mapping (Portuguese → English)

### 🧠 Initial Observation
The dataset follows a **relational structure**, similar to a real-world e-commerce database.

In [13]:
orders = dataframes.get("olist_orders_dataset")
order_items = dataframes.get("olist_order_items_dataset")
products = dataframes.get("olist_products_dataset")
customers = dataframes.get("olist_customers_dataset")
payments = dataframes.get("olist_order_payments_dataset")
reviews = dataframes.get("olist_order_reviews_dataset")
sellers = dataframes.get("olist_sellers_dataset")
geolocation = dataframes.get("olist_geolocation_dataset")
category_translation = dataframes.get("product_category_name_translation")

print("All datasets assigned successfully!")

All datasets assigned successfully!


In [17]:
for name, df in dataframes.items():
    print("\n" + "="*50)
    print(f"Dataset: {name}")
    print("="*50)
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nSample:")
    print(df.head(2))


Dataset: olist_sellers_dataset
Shape: (3095, 4)

Columns:
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Sample:
                          seller_id  seller_zip_code_prefix seller_city seller_state
0  3442f8959a84dea7ee197c632cb2df15                   13023    campinas           SP
1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844  mogi guacu           SP

Dataset: product_category_name_translation
Shape: (71, 2)

Columns:
['product_category_name', 'product_category_name_english']

Sample:
    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories

Dataset: olist_orders_dataset
Shape: (99441, 8)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Sample:
                           order_id       

In [15]:
def inspect_df(df, name):
    print("\n" + "="*60)
    print(f"Inspecting: {name}")
    print("="*60)
    
    print("\nShape:")
    print(df.shape)
    
    print("\nData Types:")
    print(df.dtypes)
    
    print("\nMissing Values:")
    print(df.isnull().sum().sort_values(ascending=False))
    
    print("\nDuplicate Rows:")
    print(df.duplicated().sum())
    
    print("\nDescribe:")
    print(df.describe(include="all").transpose())

In [16]:
inspect_df(orders, "orders")
inspect_df(order_items, "order_items")
inspect_df(products, "products")
inspect_df(customers, "customers")
inspect_df(payments, "payments")
inspect_df(reviews, "reviews")
inspect_df(sellers, "sellers")


Inspecting: orders

Shape:
(99441, 8)

Data Types:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Missing Values:
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
order_purchase_timestamp            0
order_status                        0
customer_id                         0
order_estimated_delivery_date       0
dtype: int64

Duplicate Rows:
0

Describe:
                               count unique                               top   freq
order_id                       99441  99441  66dea50a8b16d9b4dee7af250b4be1a5      1
customer_id                    99441  99441  edb027a75a1449115f6b43211ae02a24   

In [18]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

print("Datetime conversion completed!")

Datetime conversion completed!


## 🔍 Data Inspection Summary

### 🧾 General Observations
- Each dataset has different shapes and column structures
- Several tables contain **missing values**
- Some tables contain **duplicate records**
- Data types are not fully optimized (e.g., timestamps still as object)

### ⚠️ Data Quality Issues Identified
- Missing values in:
  - order timestamps
  - review data
- Potential inconsistencies in geolocation data
- Need for datetime conversion

### 🧠 Insight
Data cleaning will be a **critical next step** before analysis or modeling.

In [19]:
orders_items = orders.merge(order_items, on="order_id", how="inner")

print("Orders shape:", orders.shape)
print("Order Items shape:", order_items.shape)
print("Joined shape:", orders_items.shape)

# Check number of items per order
items_per_order = order_items.groupby("order_id").size()

print("\nMax items in a single order:", items_per_order.max())
print("Average items per order:", items_per_order.mean())

Orders shape: (99441, 8)
Order Items shape: (112650, 7)
Joined shape: (112650, 14)

Max items in a single order: 21
Average items per order: 1.1417306873695092


## 🔗 Understanding Table Relationships

### Key Relationships:
- One **customer → many orders**
- One **order → many order items**
- One **order item → one product**
- One **order item → one seller**
- One **order → multiple payments**
- One **order → one review (mostly)**

### 🧠 Insight
- The dataset represents a **marketplace system**
- Orders can contain **multiple items and sellers**
- Data granularity differs:
  - orders → transaction level
  - order_items → item level

👉 This distinction is important for accurate analysis (e.g., revenue calculation).

In [20]:
# Total Orders
total_orders = orders["order_id"].nunique()

# Total Customers
total_customers = customers["customer_id"].nunique()

# Total Revenue
total_revenue = order_items["price"].sum()

print(f"Total Orders: {total_orders}")
print(f"Total Customers: {total_customers}")
print(f"Total Revenue: {total_revenue:,.2f}")

Total Orders: 99441
Total Customers: 99441
Total Revenue: 13,591,643.70


## 📊 Initial Business Metrics

### Key Findings:
- Total number of orders gives an overview of platform activity
- Total customers indicate user base size
- Total revenue (from order_items) reflects business performance

### 🧠 Insight
- Revenue should be calculated at the **order_items level**, not orders
- This is because:
  - One order can contain multiple items
  - Each item has its own price

👉 Using the wrong table can lead to incorrect metrics.

In [23]:
orders["order_month"] = orders["order_purchase_timestamp"].dt.to_period("M")

orders_per_month = orders.groupby("order_month").size()

print(orders_per_month.head())

order_month
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
Freq: M, dtype: int64


## 📈 Orders Over Time

### Observations:
- Orders are distributed over multiple months/years
- There is a clear temporal component in the dataset

### 🧠 Insight
- The dataset supports **time-series analysis**
- Possible future analysis:
  - Monthly revenue trends
  - Seasonality patterns
  - Growth analysis

👉 This is useful for building an analytics agent that answers trend-related questions.

In [24]:
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

orders.to_csv(os.path.join(PROCESSED_DIR, "orders_clean.csv"), index=False)
order_items.to_csv(os.path.join(PROCESSED_DIR, "order_items_clean.csv"), index=False)

print("Cleaned data saved to /data/processed/")

Cleaned data saved to /data/processed/


## 💾 Data Export

Cleaned versions of selected datasets have been saved to the `processed/` directory.

### Purpose:
- Avoid reprocessing raw data repeatedly
- Prepare datasets for:
  - data cleaning
  - feature engineering
  - analytics queries

### 🧠 Insight
Separating **raw vs processed data** is a best practice in real-world data pipelines.

# 🚀 Summary & Next Steps

## ✅ What We’ve Done
- Loaded all datasets
- Inspected structure and quality
- Identified relationships between tables
- Generated initial business metrics
- Observed time-based patterns

## 🧠 Key Insights
- Dataset is highly relational (like SQL database)
- Order_items is the most important table for revenue analysis
- Data cleaning is required before deeper analysis
- Dataset supports business analytics use cases

## 🔜 Next Steps
- Data Cleaning:
  - Handle missing values
  - Fix data types
  - Remove duplicates
- Data Merging:
  - Build a unified analytics table
- Exploratory Data Analysis (EDA)
- Feature Engineering for AI/ML use cases

---

This notebook provides the foundation for building an **LLM-powered E-commerce Data Analyst Agent**.